In [ ]:
import folium
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
import io
import base64
from PIL import Image

In [ ]:
%%time 

# autoload external python modules if they changed
%load_ext autoreload
%autoreload 2
# add ../funcs to the current path
import sys, os
sys.path.append(os.path.join(os.getcwd(), "../funcs")) 

In [ ]:
%%time

# read data from netcdf files, save them to 'dataset'
from jdiag import load_jdiag, get_jdiag_metadata, get_jdiag_data
dataset = load_jdiag("../data/mpasjedi/jdiag_aircar_t133.nc")

data = get_jdiag_data(dataset, "airTemperature")

In [ ]:
%%time

from base import query_data
query_data(data)

In [ ]:
%time

latitudes = data["latitude"]
longitudes = data["longitude"]
TempOmbg = data ["ombg"]
TempObs = data["ObsValue"]
TempOman = data ["oman"]
stationIDs = data["stationIdentification"]
height = data["height"]
TempObs = TempObs - 273.15
longitudes = longitudes - 360

In [ ]:
%time

for lat, lon, temp, stationID in zip(data["latitude"], data["longitude"], data["ombg"], data["stationIdentification"]):
    if np.ma.is_masked(temp):
        continue  # skip this row if temp is masked

    popup_text = f"""
        <b>Temp:</b> {float(temp):.2f} °C<br>
        <b>Station ID:</b> {stationID}
    """

In [ ]:
%time
grid_lon, grid_lat = np.meshgrid(np.linspace(-125, -66, 300), np.linspace(25, 49, 300))
grid_temp = griddata((longitudes, latitudes), TempObs, (grid_lon, grid_lat), method='cubic')
grid_temp = np.nan_to_num(grid_temp)

In [ ]:
%time

fig, ax = plt.subplots(figsize=(8, 6))
levels = np.linspace(-60,30,35)
ax.contourf(grid_lon, grid_lat, grid_temp, levels, cmap='coolwarm', alpha = 0.5)
ax.axis('off')
buf = io.BytesIO()
plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0, transparent=True)
plt.close()
buf.seek(0)

In [ ]:
%time

img_base64 = base64.b64encode(buf.read()).decode('utf-8')
data_url = f"data:image/png;base64,{img_base64}"

In [ ]:
%time

image_bounds = [[25, -125], [49, -66]]

m = folium.Map(location=[37.5, -95], zoom_start=4, tiles='CartoDB positron')

folium.raster_layers.ImageOverlay(
    image=data_url,
    bounds=image_bounds,
    opacity=0.6,
    interactive=True,
    cross_origin=False
).add_to(m)

In [ ]:
%%time

for lat, lon, tempObs, TempOmbg, TempOman, stationID, height  in zip(latitudes, longitudes, TempObs, TempOmbg, TempOman, stationIDs, height):
    if np.ma.is_masked(tempObs) or np.ma.is_masked(TempOmbg) or np.ma.is_masked(TempOman) or np.ma.is_masked(lat) or np.ma.is_masked(lon):  
        continue  # skip this row if temp is masked   \
    
    popup_text = f"""
        <b>TempObs:</b> {float(tempObs):.2f} °C<br>
        <b>TempOmbg</b> {float(TempOmbg) :.2f} °C<br>
        <b>TempOman</b> {float(TempOman) :.2f} °C<br>
        <b>Latitude</b> {float(lat):.2f} °<br>
        <b>Longitude</b> {float(lon): .2f} °<br>
        <b>Altitude</b> {float(height):.2f} <br>
        <b>Station ID:</b> {stationID}
    """

    folium.CircleMarker(
        location=[lat, lon],
        radius= 1,
        popup=folium.Popup(popup_text, max_width=250),
        weight = 0.3,
        fill_opacity = 0.3,
        color= 'gray',
        fill=True,
        fill_color= 'lightgray'
    ).add_to(m)

In [ ]:
m


In [ ]:
#gfs hura grid temperature data 